# DuckDB Iceberg Operations

This notebook demonstrates how to use DuckDB to work with Apache Iceberg tables through the Nessie REST catalog.

In [1]:
import duckdb

# Create a DuckDB connection
conn = duckdb.connect()

# Install and load the Iceberg extension
conn.execute("INSTALL iceberg")
conn.execute("LOAD iceberg")

print("DuckDB Iceberg extension loaded successfully!")

DuckDB Iceberg extension loaded successfully!


## Attach Iceberg Catalog

Connect to the Nessie REST catalog with S3/MinIO storage.

In [ ]:
# Create a secret for S3/MinIO credentials
conn.execute("""
    CREATE SECRET minio_secret (
        TYPE S3,
        KEY_ID 'admin',
        SECRET 'password',
        REGION 'us-east-1',
        ENDPOINT 'minio:9000',
        USE_SSL false,
        URL_STYLE 'path'
    )
""")

print("S3 secret created successfully!")

In [ ]:
# Attach the Iceberg catalog using Nessie REST endpoint
conn.execute("""
    ATTACH 'iceberg_catalog' AS iceberg_db (
        TYPE ICEBERG,
        ENDPOINT '127.0.0.1:19120/iceberg/main/',
        SECRET minio_secret
    )
""")

print("Iceberg catalog attached!")

## List Schemas and Tables

In [ ]:
# List all schemas in the Iceberg catalog
schemas = conn.execute("SHOW SCHEMAS FROM iceberg_db").df()
schemas

In [ ]:
# List tables in the iot_sensor schema
tables = conn.execute("SHOW TABLES FROM iceberg_db.iot_sensor").df()
tables

## Query Iceberg Table

In [ ]:
# Count total records
count_df = conn.execute("""
    SELECT COUNT(*) as total 
    FROM iceberg_db.iot_sensor.equipment_data
""").df()
count_df

In [ ]:
# Query sample data
sample_df = conn.execute("""
    SELECT 
        timestamp,
        "temperature (°c)",
        "vibration (mm/s)",
        "pressure (pa)",
        rpm,
        "maintenance required",
        temp_change,
        vib_change
    FROM iceberg_db.iot_sensor.equipment_data 
    LIMIT 10
""").df()
sample_df

## Aggregation Query

In [ ]:
# Aggregate by day with maintenance statistics
agg_df = conn.execute("""
    SELECT 
        DATE_TRUNC('day', timestamp) AS dt,
        COUNT(CASE WHEN "maintenance required" = 1 THEN 1 END) as maintenance_count,
        COUNT(*) as total,
        MAX("temperature (°c)") AS max_temp,
        MIN("temperature (°c)") AS min_temp,
        AVG("vibration (mm/s)") AS avg_vibration
    FROM iceberg_db.iot_sensor.equipment_data
    GROUP BY DATE_TRUNC('day', timestamp)
    ORDER BY dt DESC
    LIMIT 10
""").df()
agg_df

## View Iceberg Table Snapshots

Use the `iceberg_snapshots` function to view table history.

In [ ]:
# View snapshots of the table
snapshots_df = conn.execute("""
    SELECT * 
    FROM iceberg_snapshots('iceberg_db.iot_sensor.equipment_data')
""").df()
snapshots_df

## Table Metadata

Inspect the Iceberg table metadata including schema and partition information.

In [ ]:
# Get table metadata
metadata_df = conn.execute("""
    SELECT * 
    FROM iceberg_metadata('iceberg_db.iot_sensor.equipment_data')
""").df()
metadata_df

## Time Travel Query

Query a specific snapshot of the table using the `iceberg_scan` function.

In [ ]:
# Get the first snapshot ID from the snapshots list
first_snapshot = snapshots_df.iloc[0]['snapshot_id'] if len(snapshots_df) > 0 else None

if first_snapshot:
    print(f"Querying snapshot: {first_snapshot}")
    
    # Query data from a specific snapshot
    time_travel_df = conn.execute(f"""
        SELECT COUNT(*) as count_at_snapshot
        FROM iceberg_scan('iceberg_db.iot_sensor.equipment_data', 
                         snapshot_id := {first_snapshot})
    """).df()
    display(time_travel_df)
else:
    print("No snapshots found")

## Filter and Join Operations

In [ ]:
# Filter for maintenance-required records
maintenance_df = conn.execute("""
    SELECT 
        timestamp,
        "temperature (°c)",
        "vibration (mm/s)",
        "pressure (pa)"
    FROM iceberg_db.iot_sensor.equipment_data
    WHERE "maintenance required" = 1
    ORDER BY timestamp DESC
    LIMIT 20
""").df()
maintenance_df

## Close Connection

In [ ]:
# Close the DuckDB connection
conn.close()
print("Connection closed.")